In [ ]:
# ------------------------------------------------- offline deps (zarr v3 stack)
import glob, subprocess, sys
from pathlib import Path

need = ["zarr>=3.0.10,<4", "blosc2", "numcodecs>=0.13,<0.16", "donfig>=0.8",
        "google-crc32c>=1.5", "crc32c", "deprecated", "wrapt", "typing-extensions>=4.13"]
wheel_dirs = sorted({str(Path(p).parent) for p in
                     glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True)})
assert wheel_dirs, "no wheels/ dir found in attached datasets (need the support pack)"
cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q"]
for d in wheel_dirs:
    cmd += ["--find-links", d]
r = subprocess.run(cmd + need, capture_output=True, text=True)
if r.returncode != 0:
    # tolerate specs with no wheel present; install what exists, then verify imports
    for spec in need:
        subprocess.run(cmd + [spec], capture_output=True, text=True)
import zarr, numpy
print("zarr", zarr.__version__, "OK", flush=True)


In [ ]:
# ---------------------------------------------------------------- boot
import glob, shutil, sys
from pathlib import Path

WORK = Path("/kaggle/working")
COMP = next(p for p in [Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
                        Path("/kaggle/input/biohub-cell-tracking-during-development")] if p.exists())
TRAIN_DIR = COMP / "train"
assert TRAIN_DIR.exists(), f"no train mount at {TRAIN_DIR}"
n_zarr = len(list(TRAIN_DIR.glob("*.zarr"))); n_geff = len(list(TRAIN_DIR.glob("*.geff")))
print(f"train mount: {n_zarr} zarr / {n_geff} geff", flush=True)
assert n_zarr >= 190 and n_geff >= 190

# ctrack package: dataset ships package CONTENTS (find by signature file, ctrack_test-style).
hits = [h for h in glob.glob("/kaggle/input/**/detect/train.py", recursive=True)
        if "tracking_repo" not in h]
assert hits, "ctrack-src not found under /kaggle/input"
root = Path(hits[0]).parents[1]
pkg = WORK / "src" / "ctrack"
if pkg.exists():
    shutil.rmtree(pkg)
shutil.copytree(root, pkg)
(pkg / "__init__.py").touch()
sys.path.insert(0, str(WORK / "src"))
import inspect
import ctrack.detect.train as T
sig = inspect.signature(T.train_fold)
for k in ("resume", "pseudo_dir", "seed", "iso_mode"):
    assert k in sig.parameters, f"STALE ctrack-src: '{k}' missing -- version the dataset first"
print("ctrack loaded from", root, flush=True)

# Teacher peaks.
peak_files = glob.glob("/kaggle/input/**/biohub-teacher-peaks/**/*.npz", recursive=True) or \
             glob.glob("/kaggle/input/**/teacher*peaks*/**/*.npz", recursive=True) or \
             [p for p in glob.glob("/kaggle/input/**/*.npz", recursive=True)
              if ("44b6_" in Path(p).name or "6bba_" in Path(p).name)]
assert peak_files, "no teacher-peaks npz found"
PEAKS = Path(peak_files[0]).parent
print(f"teacher peaks: {PEAKS} ({len(list(PEAKS.glob('*.npz')))} volumes)", flush=True)

# Resume seed: prior version's checkpoint, if attached.
(WORK / "models").mkdir(parents=True, exist_ok=True)
prev = sorted(glob.glob("/kaggle/input/**/checkpoint_last_fold0.pt", recursive=True))
if prev:
    shutil.copy(prev[0], WORK / "models" / "checkpoint_last_fold0.pt")
    print("RESUME SEED: copied", prev[0], flush=True)
else:
    print("fresh run (no prior checkpoint attached)", flush=True)


In [ ]:
# ---------------------------------------------------------------- train (dettrain-s0)
from ctrack.detect.train import train_fold

best = train_fold(
    str(TRAIN_DIR), str(TRAIN_DIR),
    fold=0, k=5,
    epochs=60,
    batch=8,
    base=32,
    lr=1e-3,
    train_stride=2,
    val_stride=6,
    pos_weight=4.0,
    out_dir=str(WORK / "models"),
    limit=None,
    num_workers=2,
    log_every=50,
    max_minutes=640,
    early_stop_patience=8,
    resume="auto",
    pseudo_dir=str(PEAKS),
    pseudo_weight=0.3,
    seed=0,
    iso_mode="stride",
)
print("BEST SEL:", best, flush=True)
